# 06.2 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/02-kmeans-by-hand/exercises.ipynb)

Do exercises 1 and 2 on paper first, then check with code.

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

In [ ]:
names = list("ABCDEFGH")
points = np.array([[1, 1], [1, 3], [2, 2], [4, 3],      # x = orders last year
                   [6, 6], [7, 5], [7, 8], [9, 7]],     # y = average items per order
                  dtype=float)
pd.DataFrame(points, index=names, columns=["orders", "items"]).T

In [ ]:
def kmeans_by_hand(points, centroids, max_iter=10):
    """Plain K-Means: assign to nearest centroid, move centroids to the mean, repeat."""
    labels = None
    for it in range(1, max_iter + 1):
        dist = np.sqrt(((points[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2))
        new_labels = dist.argmin(axis=1)                         # assignment step
        inertia = ((points - centroids[new_labels]) ** 2).sum()
        yield it, centroids.copy(), dist, new_labels, inertia
        if labels is not None and (new_labels == labels).all():
            return                                               # nothing changed: converged
        labels = new_labels
        centroids = np.array([points[labels == j].mean(axis=0)   # update step
                              for j in range(len(centroids))])

## Exercise 1: a different start

Start the two centroids on **G (7, 8)** and **H (9, 7)**. Run K-Means by hand. How many iterations does it take? Do you reach the same final clusters as in the lesson? Check with `kmeans_by_hand`.

In [ ]:
# your code here

## Exercise 2: a worse split

Put {A, B, C} in one cluster and {D, E, F, G, H} in the other, with each centroid at its cluster's mean. Compute the inertia by hand, then in code. Is it higher or lower than 18.5?

In [ ]:
# your code here

## Exercise 3: inertia for every k

Run `KMeans(n_clusters=k, n_init=10, random_state=42)` for k = 1 to 8 on the eight customers and print the inertia. What is it at k = 8, and why?

In [ ]:
# your code here

## Exercise 4: scaling without the log

Build the RFM table, standardise it **without** taking logs first, and run K-Means with k = 4. Compare the cluster sizes with the lesson's raw and log-scaled versions.

In [ ]:
import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

snapshot = sales["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)    # 2011-12-10
rfm = sales.groupby("Customer ID").agg(
    recency=("InvoiceDate", lambda d: (snapshot - d.max()).days),
    frequency=("Invoice", "nunique"),
    monetary=("Revenue", "sum"))
print(f"{len(sales):,} rows, {len(rfm):,} customers, snapshot {snapshot.date()}")

In [ ]:
# your code here